# ESP32 로봇 강아지 서보 조정

이 노트북은 `esp32-robot-dog-code-master/tools/servoInitPose/servoInitPose.ino` 펌웨어와 함께 사용합니다. LF/GAMMA 핀을 바꾸려면 이 펌웨어를 다시 업로드해야 합니다. 핀 변경 기능은 ESP32 Arduino core 3.x에서 동작합니다.

## 안전
- 로봇을 단단한 받침대에 고정하고 네 발이 공중에 뜨게 합니다. 관절과 혼의 간섭을 확인하고 전원 차단 스위치를 손 닿는 곳에 둡니다.
- USB 연결만으로는 서보 전원이 공급되지 않을 수 있습니다. 처음 연결/통신 확인 때는 서보 전원을 끄고, 움직임 준비가 끝난 후에만 켭니다.
- 모의 테스트 셀은 포트를 열지 않습니다. 실제 보드 제어 셀을 실행해 UI를 띄운 뒤 Arduino IDE의 시리얼 모니터/플로터를 닫고 연결 버튼을 누르세요. 움직임 활성화 버튼을 누르기 전에는 이동 명령을 전송하지 않습니다.
- 펌웨어와 노트북은 초기 관절값(α/β/γ) 90/45/90도에서 관절별 ±15도만 허용합니다. 한 번의 `SET` 변화도 최대 5도입니다. 걸림/떨림/소음이 나면 즉시 서보 전원을 끕니다.
- LF/GAMMA 핀 확인용 선택지는 기본 GPIO 27과 다른 서보에 할당되지 않은 후보 GPIO 32/33입니다. 핀 변경은 재부팅하면 GPIO 27로 돌아옵니다. 핀 변경 후에는 서보 전원을 끄고 LF/GAMMA의 신호선을 선택한 GPIO로 옮긴 다음 전원을 켜세요. GPIO 번호와 보드 실크 인쇄가 맞는지 확인하고 다른 장치에 이미 연결된 핀은 사용하지 마세요.
- 이 값은 관절 좌표입니다. 서보축의 실제 각도는 좌우 방향과 기어비에 따라 다릅니다. 고장이나 간섭이 생기면 프로그램의 범위 제한에만 의존하지 마세요.

## 실제 보드 연결 및 모터 테스트

이 셀은 앞의 설정 셀을 실행하지 않아도 되도록 필요한 설정과 제어 코드를 포함합니다. 선택한 Python 커널에 `ipywidgets` 또는 `pyserial`이 없으면 이 셀이 자동 설치한 뒤 계속 실행합니다. 인터넷 연결과 pip 사용이 필요하며, 설치 실패 시 오류를 표시하고 COM 포트는 열지 않습니다.

이 셀은 모터 테스트 UI만 표시합니다. Arduino IDE의 시리얼 모니터/플로터를 닫은 다음 COM14 연결 버튼을 누르세요. 연결 시 DTR/RTS를 끈 채로 포트를 열어 ESP32 리셋을 막습니다(USB-시리얼 드라이버에 따라 리셋될 수 있음). 리셋되더라도 펌웨어가 12개 서보를 즉시 중립 자세(90/45/90)로 구동합니다. 서보 전원은 안전 확인 전까지 끄고 로봇을 단단히 고정하세요.

보드 연결 후 움직임 활성화 버튼을 눌러야 슬라이더가 활성화됩니다. 활성화 후에는 슬라이더를 움직이면 선택한 관절이 바로 움직입니다. 범위는 초기 각도 기준 ±15도이며 한 명령은 최대 5도씩 이동합니다. 작은 각도 변화부터 시험하세요.

In [1]:
import html
import importlib.util
import subprocess
import sys
import time
from dataclasses import dataclass

required_packages = (("ipywidgets", "ipywidgets"), ("serial", "pyserial"))
missing_packages = [package for module, package in required_packages if importlib.util.find_spec(module) is None]
if missing_packages:
    print("Installing missing packages into this notebook's Python environment:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    importlib.invalidate_caches()

import ipywidgets as widgets
import serial
from IPython.display import display, clear_output

LEGS = ("LF", "RF", "LH", "RH")
JOINTS = ("ALPHA", "BETA", "GAMMA")
NEUTRAL = {"ALPHA": 90, "BETA": 45, "GAMMA": 90}
MAX_OFFSET_DEG = 15
MAX_STEP_DEG = 5

@dataclass(frozen=True)
class ServoConfig:
    pin: int
    ratio: float
    inverse: bool

SERVO_CONFIG = {
    ("LF", "ALPHA"): ServoConfig(25, 2, False),
    ("LF", "BETA"): ServoConfig(26, 1, True),
    ("LF", "GAMMA"): ServoConfig(27, 1, True),
    ("RF", "ALPHA"): ServoConfig(16, 2, True),
    ("RF", "BETA"): ServoConfig(18, 1, False),
    ("RF", "GAMMA"): ServoConfig(17, 1, False),
    ("LH", "ALPHA"): ServoConfig(13, 2, True),
    ("LH", "BETA"): ServoConfig(12, 1, True),
    ("LH", "GAMMA"): ServoConfig(14, 1, True),
    ("RH", "ALPHA"): ServoConfig(4, 2, False),
    ("RH", "BETA"): ServoConfig(2, 1, False),
    ("RH", "GAMMA"): ServoConfig(15, 1, False),
}

movement_enabled = False
previous_esp = globals().get("esp")
if previous_esp is not None and getattr(previous_esp, "is_open", False):
    previous_esp.close()
esp = None

def validate_angle(joint, angle):
    if joint not in NEUTRAL:
        raise ValueError(f"Unknown joint: {joint}")
    if isinstance(angle, bool) or not isinstance(angle, int):
        raise TypeError("Joint angles must be integer degrees")
    center = NEUTRAL[joint]
    if not center - MAX_OFFSET_DEG <= angle <= center + MAX_OFFSET_DEG:
        raise ValueError(f"{joint} is limited to {center-MAX_OFFSET_DEG}..{center+MAX_OFFSET_DEG} degrees")

def build_set_command(leg, joint, angle):
    leg, joint = leg.upper(), joint.upper()
    if leg not in LEGS or (leg, joint) not in SERVO_CONFIG:
        raise ValueError("Unknown leg or joint")
    validate_angle(joint, angle)
    return f"SET {leg} {joint} {angle}"

def open_serial_without_reset(port, baud):
    # Keep DTR/RTS deasserted while opening so the ESP32 auto-reset circuit
    # does not reboot the board (a reboot drops all servo pulses).
    port_handle = serial.Serial()
    port_handle.port = port
    port_handle.baudrate = baud
    port_handle.timeout = 0.25
    port_handle.write_timeout = 2
    port_handle.dsrdtr = False
    port_handle.rtscts = False
    port_handle.dtr = False
    port_handle.rts = False
    port_handle.open()
    return port_handle

def connect_esp32(port_name="COM14", baud=115200):
    global esp
    if not port_name.strip():
        raise ValueError("Enter a COM port, for example COM14")
    candidate = open_serial_without_reset(port_name, baud)
    connected = False
    try:
        deadline = time.monotonic() + 12
        next_hello = 0.0
        ready = False
        startup_lines = []
        while time.monotonic() < deadline:
            now = time.monotonic()
            if now >= next_hello:
                candidate.write(b"HELLO\r\n")
                next_hello = now + 1
            line = candidate.readline().decode("ascii", errors="replace").strip()
            if not line:
                continue
            startup_lines.append(line)
            if line == "READY SERVO_POSE_V1":
                ready = True
                break
            if line.startswith("ERROR:") or line == "ERR servos_not_ready":
                raise RuntimeError(f"ESP32 firmware failed to initialize: {line}")

        if not ready:
            details = "; ".join(startup_lines[-5:]) or "no serial output"
            raise RuntimeError(
                f"No READY SERVO_POSE_V1 response from {port_name}. "
                f"Last serial output: {details}. Check the firmware upload, COM port, "
                "baud rate (115200), and that no other app is using the port."
            )

        candidate.reset_input_buffer()
        candidate.write(b"STATUS\r\n")
        deadline = time.monotonic() + 3
        status = ""
        while time.monotonic() < deadline:
            line = candidate.readline().decode("ascii", errors="replace").strip()
            if line.startswith("STATUS "):
                status = line
                break
            if line.startswith("ERR") or line.startswith("ERROR:"):
                raise RuntimeError(f"ESP32 rejected STATUS: {line}")
        if not status:
            raise RuntimeError("ESP32 replied READY but did not return STATUS. Check servo firmware startup.")
        esp = candidate
        connected = True
        return status
    finally:
        if not connected:
            candidate.close()

def send_command(command, require_motion=False):
    if esp is None or not esp.is_open:
        raise RuntimeError("ESP32 is not connected")
    if require_motion and not movement_enabled:
        raise RuntimeError("Motion is disabled. Confirm ARM first.")
    expected_prefix = {
        "STATUS": "STATUS ",
        "CENTER": "OK CENTER",
        "SET": "OK SET ",
        "POSE": "OK POSE",
        "PIN": "OK PIN ",
        "SETTLED": "SETTLED ",
    }.get(command.split(maxsplit=1)[0])
    esp.write((command + "\r\n").encode("ascii"))
    deadline = time.monotonic() + 3
    while time.monotonic() < deadline:
        reply = esp.readline().decode("ascii", errors="replace").strip()
        if not reply:
            continue
        if reply.startswith("ERR") or reply.startswith("ERROR:"):
            raise RuntimeError(reply)
        if expected_prefix is None or reply.startswith(expected_prefix):
            return reply
    raise TimeoutError(f"No matching response for {command!r}")

def parse_status(status):
    if not status.startswith("STATUS "):
        raise RuntimeError(f"Unexpected status response: {status}")
    return dict(item.split("=", 1) for item in status.split()[1:])

def move_joint(leg, joint, target, step=MAX_STEP_DEG, pause_s=0.05):
    if not movement_enabled:
        raise RuntimeError("Click Enable motion before moving a servo")
    if not isinstance(step, int) or not 1 <= step <= MAX_STEP_DEG:
        raise ValueError(f"step must be 1..{MAX_STEP_DEG} degrees")
    leg, joint = leg.upper(), joint.upper()
    build_set_command(leg, joint, target)
    fields = parse_status(send_command("STATUS"))
    if leg not in fields:
        raise RuntimeError(f"Missing leg in STATUS response: {leg}")
    angles = [int(value) for value in fields[leg].split("/")]
    current = angles[JOINTS.index(joint)]
    direction = 1 if target >= current else -1
    while current != target:
        current += direction * min(step, abs(target - current))
        reply = send_command(build_set_command(leg, joint, current), require_motion=True)
        if not reply.startswith("OK SET "):
            raise RuntimeError(f"Unexpected response: {reply}")
        time.sleep(pause_s)
    return send_command("STATUS")

def arm_motion():
    global movement_enabled
    if esp is None or not esp.is_open:
        raise RuntimeError("ESP32 is not connected")
    movement_enabled = True

def wait_for_settled(timeout_s=6.0):
    # Close the port only after the servos reached their targets and the pose is
    # saved in flash. If the ESP32 resets later (USB driver, brownout), it then
    # restarts from the real servo positions instead of jumping at full speed.
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if send_command("SETTLED") == "SETTLED 1":
            return True
        time.sleep(0.2)
    return False

def disconnect_esp32():
    global esp, movement_enabled
    movement_enabled = False
    pose_saved = False
    if esp is not None and esp.is_open:
        try:
            pose_saved = wait_for_settled()
        except (RuntimeError, TimeoutError, OSError):
            pose_saved = False
        esp.close()
    esp = None
    return pose_saved

# Import dependencies before opening the serial port.
port_status = widgets.HTML(value="Not connected. Close Arduino Serial Monitor, then click Connect COM14.")
connect_button = widgets.Button(description="Connect COM14", button_style="info")
leg_selector = widgets.Dropdown(options=LEGS, description="\uB2E4\uB9AC:")
joint_selector = widgets.Dropdown(options=JOINTS, description="\uAD00\uC808:")
pin_info = widgets.HTML()
test_pin_selector = widgets.Dropdown(
    options=(("기본 GPIO 27", 27), ("테스트 GPIO 32", 32), ("테스트 GPIO 33", 33)),
    value=SERVO_CONFIG[("LF", "GAMMA")].pin,
    description="LF GAMMA 핀:",
)
test_pin_button = widgets.Button(description="테스트 핀 적용", button_style="warning", disabled=True)
angle_slider = widgets.IntSlider(description="\uBAA9\uD45C \uAC01\uB3C4:", continuous_update=True, readout=True, layout=widgets.Layout(width="520px"))
arm_button = widgets.Button(description="\uC6C0\uC9C1\uC784 \uD65C\uC131\uD654", button_style="warning")
disarm_button = widgets.Button(description="\uBE44\uD65C\uC131\uD654", disabled=True)
move_button = widgets.Button(description="\uC120\uD0DD\uD55C \uC11C\uBCF4 \uC6C0\uC9C1\uC774\uAE30", button_style="success", disabled=True)
center_button = widgets.Button(description="12\uAC1C \uC11C\uBCF4 \uC911\uB9BD \uBCF5\uADC0", disabled=True)
disconnect_button = widgets.Button(description="\uC5F0\uACB0 \uD574\uC81C")
ui_output = widgets.Output()

def show_result(action):
    with ui_output:
        clear_output(wait=True)
        try:
            result = action()
        except (RuntimeError, ValueError, TypeError, TimeoutError, OSError) as exc:
            print(f"\uC624\uB958: {exc}")
        else:
            if result is not None:
                print(result)

slider_syncing = False

def refresh_selected_control(status=None):
    global slider_syncing
    leg, joint = leg_selector.value, joint_selector.value
    center = NEUTRAL[joint]
    lower, upper = center - MAX_OFFSET_DEG, center + MAX_OFFSET_DEG
    if lower > angle_slider.max:
        angle_slider.max, angle_slider.min = upper, lower
    elif upper < angle_slider.min:
        angle_slider.min, angle_slider.max = lower, upper
    else:
        angle_slider.min, angle_slider.max = lower, upper
    slider_syncing = True
    try:
        angle_slider.value = int(parse_status(status or send_command("STATUS"))[leg].split("/")[JOINTS.index(joint)]) if esp is not None and esp.is_open else center
    finally:
        slider_syncing = False
    pin_info.value = f"<b>{leg} / {joint} - GPIO {SERVO_CONFIG[(leg, joint)].pin}</b>"

def update_enabled_state():
    connected = esp is not None and esp.is_open
    armed = connected and movement_enabled
    connect_button.disabled = connected
    arm_button.disabled = not connected or armed
    disarm_button.disabled = not armed
    angle_slider.disabled = not armed
    move_button.disabled = not armed
    center_button.disabled = not armed
    disconnect_button.disabled = not connected
    test_pin_selector.disabled = not connected or armed
    test_pin_button.disabled = (
        not connected or armed or leg_selector.value != "LF" or
        joint_selector.value != "GAMMA" or
        test_pin_selector.value == SERVO_CONFIG[("LF", "GAMMA")].pin
    )

def on_connect(_):
    connect_button.disabled = True
    port_status.value = "Connecting to COM14..."
    try:
        status = connect_esp32("COM14")
        refresh_selected_control(status)
        port_status.value = "<b>Connected to COM14.</b> Click Enable motion before moving."
        update_enabled_state()
        with ui_output:
            clear_output(wait=True)
            print("Connected to ESP32; STATUS received.")
    except (RuntimeError, ValueError, OSError) as exc:
        port_status.value = f"<b>Connection failed:</b> {html.escape(str(exc))}"
        connect_button.disabled = False
        with ui_output:
            clear_output(wait=True)
            print(f"Connection error: {exc}")

def on_arm(_):
    def arm_from_ui():
        arm_motion()
        update_enabled_state()
        return "\uC6C0\uC9C1\uC784\uC774 \uD65C\uC131\uD654\uB418\uC5C8\uC2B5\uB2C8\uB2E4. 1\uB3C4 \uBCC0\uD654\uBD80\uD130 \uC2DC\uC791\uD558\uC138\uC694."
    show_result(arm_from_ui)

def on_disarm(_):
    global movement_enabled
    movement_enabled = False
    update_enabled_state()
    with ui_output:
        clear_output(wait=True)
        print("\uC6C0\uC9C1\uC784\uC774 \uBE44\uD65C\uC131\uD654\uB418\uC5C8\uC2B5\uB2C8\uB2E4. \uC11C\uBCF4\uC758 \uD604\uC7AC \uC704\uCE58\uB294 \uC720\uC9C0\uB429\uB2C8\uB2E4.")

def on_move(_):
    def move_selected():
        leg, joint, target = leg_selector.value, joint_selector.value, angle_slider.value
        status = move_joint(leg, joint, target)
        refresh_selected_control(status)
        return f"{leg} / {joint} (GPIO {SERVO_CONFIG[(leg, joint)].pin}) moved to {target} degrees."
    show_result(move_selected)

def on_angle_change(change):
    if slider_syncing or change.get("name") != "value" or not movement_enabled:
        return
    def move_to_slider_value():
        leg, joint, target = leg_selector.value, joint_selector.value, change["new"]
        status = move_joint(leg, joint, target)
        refresh_selected_control(status)
        return f"{leg} / {joint} (GPIO {SERVO_CONFIG[(leg, joint)].pin}) moved to {target} degrees."
    show_result(move_to_slider_value)

def on_center(_):
    def center_all():
        reply = send_command("CENTER", require_motion=True)
        if reply != "OK CENTER":
            raise RuntimeError(f"Unexpected response: {reply}")
        time.sleep(1.5)
        refresh_selected_control(send_command("STATUS"))
        return "12\uAC1C \uC11C\uBCF4\uB97C \uC911\uB9BD \uC790\uC138\uB85C \uBCF5\uADC0\uD588\uC2B5\uB2C8\uB2E4."
    show_result(center_all)

def on_selection_change(_):
    refresh_selected_control()
    update_enabled_state()

def on_test_pin(_):
    def apply_test_pin():
        if leg_selector.value != "LF" or joint_selector.value != "GAMMA":
            raise ValueError("Pin changes are only supported for LF / GAMMA")
        if movement_enabled:
            raise RuntimeError("Disarm motion before changing the test pin")
        pin = test_pin_selector.value
        reply = send_command(f"PIN LF GAMMA {pin}")
        if reply != f"OK PIN LF GAMMA {pin}":
            raise RuntimeError(f"Unexpected pin response: {reply}")
        current = SERVO_CONFIG[("LF", "GAMMA")]
        SERVO_CONFIG[("LF", "GAMMA")] = ServoConfig(pin, current.ratio, current.inverse)
        refresh_selected_control(send_command("STATUS"))
        update_enabled_state()
        return f"LF / GAMMA signal output changed to GPIO {pin}. Power off the servo before moving its signal wire."
    show_result(apply_test_pin)

def on_disconnect(_):
    pose_saved = disconnect_esp32()
    update_enabled_state()
    with ui_output:
        clear_output(wait=True)
        if not pose_saved:
            print("서보 자세 저장을 확인하지 못했습니다. 다음 접속 때 서보가 빠르게 움직일 수 있습니다.")
        print("\uC5F0\uACB0\uC744 \uD574\uC81C\uD588\uC2B5\uB2C8\uB2E4. \uC11C\uBCF4 \uC804\uC6D0\uC740 \uBCC4\uB3C4\uB85C \uB04A\uC73C\uC138\uC694.")

def on_close(_):
    disconnect_esp32()
    update_enabled_state()

connect_button.on_click(on_connect)
arm_button.on_click(on_arm)
disarm_button.on_click(on_disarm)
move_button.on_click(on_move)
center_button.on_click(on_center)
disconnect_button.on_click(on_disconnect)
leg_selector.observe(on_selection_change, names="value")
joint_selector.observe(on_selection_change, names="value")
test_pin_selector.observe(lambda _: update_enabled_state(), names="value")
test_pin_button.on_click(on_test_pin)
angle_slider.observe(on_angle_change, names="value")
refresh_selected_control()
update_enabled_state()
display(widgets.VBox([
    port_status,
    widgets.HBox([connect_button, disconnect_button]),
    widgets.HTML("Secure the robot with its feet off the ground. After enabling motion, moving the slider sends commands immediately."),
    widgets.HBox([leg_selector, joint_selector, pin_info]),
    widgets.HBox([test_pin_selector, test_pin_button]),
    angle_slider,
    widgets.HBox([arm_button, disarm_button]),
    widgets.HBox([move_button, center_button, disconnect_button]),
    ui_output,
]))
